# Phase 1 — Data Inspection & Understanding

This notebook only inspects and documents the two original datasets. It does not merge, fill, encode, drop, or train anything.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")

## 2. Load the Datasets

The original CSV files are read without modification.

In [ ]:
from pathlib import Path

project = Path.cwd().parent
transaction_path = project / "data" / "raw" / "train_transaction.csv"
identity_path = project / "data" / "raw" / "train_identity.csv"

train_transaction = pd.read_csv(transaction_path, low_memory=False)
train_identity = pd.read_csv(identity_path, low_memory=False)

print("Transaction file:", transaction_path)
print("Identity file:", identity_path)
print("Original files were not modified.")

## 3. Basic Dataset Information

**What was checked?** Shape, first and last rows, data types, and memory usage.

**Meaning:** These checks establish the size and structure before any later processing.

**Decision:** Do not modify the datasets at this stage.

In [ ]:
for name, df in [("train_transaction", train_transaction), ("train_identity", train_identity)]:
    print("\n=====", name, "=====")
    print("Shape:", df.shape)
    print("Head:")
    display(df.head())
    print("Tail:")
    display(df.tail())
    print("Data types:")
    display(df.dtypes.value_counts())
    print("Memory usage:", round(df.memory_usage(deep=True).sum() / 1024**2, 2), "MB")

In [ ]:
summary = pd.DataFrame({
    "Dataset": ["train_transaction", "train_identity"],
    "Rows": [len(train_transaction), len(train_identity)],
    "Columns": [len(train_transaction.columns), len(train_identity.columns)]
})
display(summary)

## 4. Transaction Dataset Analysis

### 4.1 Column Names and Types

All column names are available through `train_transaction.columns`. A compact grouped display keeps the notebook readable.

In [ ]:
print("Transaction columns:")
for start in range(0, len(train_transaction.columns), 50):
    print(train_transaction.columns[start:start+50].tolist())

print("\nData types:")
display(train_transaction.dtypes.value_counts())
numerical_columns = train_transaction.select_dtypes(include=np.number).columns.tolist()
categorical_columns = train_transaction.select_dtypes(include=["object", "category"]).columns.tolist()
boolean_columns = train_transaction.select_dtypes(include="bool").columns.tolist()
print("Numerical:", len(numerical_columns))
print("Categorical/Object:", len(categorical_columns))
print("Boolean:", len(boolean_columns))

### 4.2 Numerical and Categorical Features

In [ ]:
print("Numerical statistics:")
display(train_transaction[numerical_columns].describe().T.head(20))

print("Categorical columns:", categorical_columns)
for column in categorical_columns:
    print("\n", column, "unique values:", train_transaction[column].nunique(dropna=False))
    display(train_transaction[column].value_counts(dropna=False).head(10))

**Interpretation:** Numerical columns have different scales and ranges. Categorical columns have limited or moderate numbers of values. No encoding or type conversion is performed now.

## 5. Identity Dataset Analysis

In [ ]:
identity_numerical = train_identity.select_dtypes(include=np.number).columns.tolist()
identity_categorical = train_identity.select_dtypes(include=["object", "category"]).columns.tolist()
print("Columns:", train_identity.columns.tolist())
print("Numerical columns:", len(identity_numerical))
print("Categorical/Object columns:", len(identity_categorical))
print("Identity numerical statistics:")
display(train_identity[identity_numerical].describe().T)

for column in ["DeviceInfo"] + [c for c in train_identity.columns if c.startswith("id_")]:
    if column in train_identity.columns:
        print("\n", column, "unique values:", train_identity[column].nunique(dropna=False))
        display(train_identity[column].value_counts(dropna=False).head(10))

**Interpretation:** Identity and device fields contain substantial missingness and some high-cardinality values. They are recorded for later investigation; they are not removed or transformed in Phase 1.

## 6. Target Analysis

In [ ]:
target_counts = train_transaction["isFraud"].value_counts(dropna=False)
target_percent = train_transaction["isFraud"].value_counts(normalize=True, dropna=False) * 100
print("Counts:")
display(target_counts)
print("Percentages:")
display(target_percent)
print("Total transactions:", len(train_transaction))
print("Legitimate (0):", int(target_counts.get(0, 0)))
print("Fraud (1):", int(target_counts.get(1, 0)))
print("Observed values:", sorted(train_transaction["isFraud"].dropna().unique().tolist()))

In [ ]:
sns.countplot(x="isFraud", data=train_transaction)
plt.title("Fraud vs Legitimate Transactions")
plt.xlabel("isFraud")
plt.ylabel("Number of transactions")
plt.show()

**Finding:** The target is binary with values 0 and 1. In this dataset, 0 represents legitimate transactions and 1 represents fraud, consistent with the dataset definition.

**Impact:** Fraud is the minority class, so later evaluation must not rely on accuracy alone. No imbalance method is applied now.

## 7. TransactionID Analysis

The datasets are not merged in this phase.

In [ ]:
for name, df in [("Transaction", train_transaction), ("Identity", train_identity)]:
    print(name)
    print("Exists:", "TransactionID" in df.columns)
    print("Missing:", int(df["TransactionID"].isna().sum()))
    print("Unique:", int(df["TransactionID"].nunique()))
    print("Duplicates:", int(df["TransactionID"].duplicated().sum()))

transaction_ids = set(train_transaction["TransactionID"].dropna())
identity_ids = set(train_identity["TransactionID"].dropna())
common_ids = transaction_ids.intersection(identity_ids)
print("\nTransaction IDs:", len(transaction_ids))
print("Identity IDs:", len(identity_ids))
print("IDs in both:", len(common_ids))
print("Transaction IDs without identity:", len(transaction_ids - identity_ids))

**Interpretation:** `TransactionID` exists in both datasets, has no missing values or duplicates in either dataset, and is suitable for controlled future joining. The actual merge is intentionally postponed.

## 8. Missing Value Analysis

In [ ]:
def missing_table(df):
    table = pd.DataFrame({
        "Feature": df.columns,
        "Missing Count": df.isnull().sum().values,
        "Missing %": (df.isnull().mean().values * 100)
    })
    return table.sort_values("Missing %", ascending=False)

transaction_missing = missing_table(train_transaction)
identity_missing = missing_table(train_identity)
print("Transaction total missing cells:", int(train_transaction.isnull().sum().sum()))
print("Identity total missing cells:", int(train_identity.isnull().sum().sum()))
print("Top transaction missing features:")
display(transaction_missing.head(20))
print("Top identity missing features:")
display(identity_missing.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=transaction_missing.head(20), y="Feature", x="Missing %", ax=axes[0], color="steelblue")
axes[0].set_title("Top 20 Missing Transaction Features")
sns.barplot(data=identity_missing.head(20), y="Feature", x="Missing %", ax=axes[1], color="darkorange")
axes[1].set_title("Top 20 Missing Identity Features")
plt.tight_layout()
plt.show()

**Interpretation:** Missing information is substantial, especially in selected transaction and identity fields. No values are filled and no columns are removed. Later preprocessing must decide how to handle missingness using training data only.

## 9. Duplicate Analysis

In [ ]:
print("Transaction duplicate rows:", int(train_transaction.duplicated().sum()))
print("Identity duplicate rows:", int(train_identity.duplicated().sum()))
print("Transaction duplicate TransactionIDs:", int(train_transaction["TransactionID"].duplicated().sum()))
print("Identity duplicate TransactionIDs:", int(train_identity["TransactionID"].duplicated().sum()))

**Interpretation:** Duplicate full rows and duplicate keys are checked but not deleted. If duplicates appear later, they may affect joining and model evaluation.

## 10. Cardinality Analysis

For this initial audit, low cardinality means ≤10 unique values, medium means 11–100, and high means >100. This is only a descriptive classification.

In [ ]:
def cardinality_table(df):
    columns = df.select_dtypes(include=["object", "category"]).columns
    table = pd.DataFrame({
        "Feature": columns,
        "Data Type": [str(df[c].dtype) for c in columns],
        "Unique Values": [df[c].nunique(dropna=False) for c in columns]
    })
    table["Cardinality"] = pd.cut(table["Unique Values"], bins=[-1, 10, 100, np.inf], labels=["Low", "Medium", "High"])
    return table.sort_values("Unique Values", ascending=False)

transaction_cardinality = cardinality_table(train_transaction)
identity_cardinality = cardinality_table(train_identity)
print("Transaction categorical cardinality:")
display(transaction_cardinality)
print("Identity categorical cardinality:")
display(identity_cardinality)

**Interpretation:** High-cardinality fields may require careful later encoding. No encoding is performed in this phase.

## 11. Initial Data Quality Assessment

In [ ]:
transaction_constant = train_transaction.nunique(dropna=False) == 1
identity_constant = train_identity.nunique(dropna=False) == 1
transaction_identifier_candidates = [c for c in train_transaction.columns if c.lower() == "transactionid" or c.lower().startswith("card") or c.lower().endswith("id")]
identity_identifier_candidates = [c for c in train_identity.columns if c.lower() == "transactionid" or c.lower().startswith("id_") or "device" in c.lower()]

print("Constant transaction features:", train_transaction.columns[transaction_constant].tolist())
print("Constant identity features:", train_identity.columns[identity_constant].tolist())
print("Potential transaction identifiers:", transaction_identifier_candidates)
print("Potential identity/device identifiers:", identity_identifier_candidates)
print("\nPotential risks identified:")
print("- Missing data: substantial missingness requires later, train-only handling.")
print("- High cardinality: some identity/device fields may need careful encoding later.")
print("- Identifier leakage: ID-like fields must be investigated before modeling.")
print("- Memory: the transaction file is wide and requires memory-aware processing.")
print("- Duplicate risk: currently no duplicate rows or keys were observed." )

**Identifier assessment:** `TransactionID` is primarily a join key, not automatically a useful predictive feature. Card and identity/device fields may contain useful behavioral information, but their predictive value and leakage risk require later investigation. No identifier is removed now.

## 12. Phase 1 Conclusion

### Phase 1 — Final Data Inspection Report

1. **Dataset Overview:** Both original CSV datasets are readable and contain the expected transaction and identity information.
2. **Transaction Dataset:** The dataset is wide, contains numerical and categorical fields, and has substantial missingness.
3. **Identity Dataset:** Identity/device fields are partially populated and include high-cardinality columns.
4. **Target Analysis:** `isFraud` is binary with observed values 0 and 1; the class distribution is imbalanced.
5. **TransactionID Analysis:** The key exists in both datasets, is non-missing and unique in each, and can be considered for a controlled future join.
6. **Missing Values:** Missingness is a major data-quality characteristic; no filling or deletion was performed.
7. **Duplicates:** Duplicate rows and duplicate keys were checked and not deleted.
8. **Cardinality:** Categorical cardinality was measured descriptively; no encoding was performed.
9. **Constant Features:** Constant fields were checked and not removed.
10. **Potential Identifier Features:** ID-like fields were identified for later investigation, not automatically discarded.
11. **Integration Readiness:** The datasets are structurally ready for a carefully validated future merge, but the merge is not performed here.
12. **Potential Risks:** Missing data, class imbalance, high cardinality, identifier leakage, memory usage, and possible distribution differences between records with and without identity information.
13. **Recommended Next Step:** After explicit approval, create `02_data_integration.ipynb` and perform a controlled merge using `TransactionID`, with row-count and key-integrity checks.

**Decision:** Stop after Phase 1. Do not start data integration or any modeling phase yet.